# 22 · Testing with pytest

Untested pipelines break silently and corrupt data. **Tests** encode what
"correct" means and catch regressions. `pytest` is the standard: plain `assert`
statements, powerful fixtures, and parametrization. We run tests inside the
notebook with `ipytest`.

In [ ]:
import ipytest
ipytest.autoconfig()
print('ipytest ready')

## The function under test

Realistic testing means testing your **transforms**. Here's a small, pure
cleaning function — pure functions are the easiest and most valuable to test.

In [ ]:
def clean_country(value):
    '''Normalize a country code: strip, upper, blank -> UNKNOWN.'''
    if value is None:
        return 'UNKNOWN'
    v = value.strip().upper()
    return v or 'UNKNOWN'

print(clean_country('  us '), '|', clean_country(''), '|', clean_country(None))

## Writing tests: just `assert`

A pytest test is a function named `test_*` containing `assert`s. `pytest`
introspects a failing assert to show exactly what differed.

In [ ]:
%%ipytest

def test_strips_and_uppercases():
    assert clean_country('  us ') == 'US'

def test_blank_becomes_unknown():
    assert clean_country('') == 'UNKNOWN'

def test_none_becomes_unknown():
    assert clean_country(None) == 'UNKNOWN'

## Parametrize: many cases, one test

`@pytest.mark.parametrize` runs the same test over many input/output pairs —
concise coverage of edge cases.

In [ ]:
%%ipytest

import pytest

@pytest.mark.parametrize('raw, expected', [
    ('us', 'US'),
    ('  GB', 'GB'),
    ('de  ', 'DE'),
    ('', 'UNKNOWN'),
    (None, 'UNKNOWN'),
])
def test_clean_country(raw, expected):
    assert clean_country(raw) == expected

## Fixtures: reusable test setup

A **fixture** builds shared test data or resources (sample rows, a temp
database) and hands it to any test that names it as an argument. This keeps
tests clean and isolated. Here the rows are plain dicts — the same idea applies
to a DataFrame or a database connection.

In [ ]:
%%ipytest

import pytest

def revenue_by_status(rows):
    return sum(r['amount'] for r in rows if r['status'] == 'completed')

@pytest.fixture
def sample_orders():
    return [
        {'amount': 100.0, 'status': 'completed'},
        {'amount': 50.0, 'status': 'returned'},
        {'amount': 25.0, 'status': 'completed'},
    ]

def test_revenue_only_counts_completed(sample_orders):
    assert revenue_by_status(sample_orders) == 125.0

def test_revenue_empty_is_zero():
    assert revenue_by_status([]) == 0

## Testing that errors are raised

Assert that bad input raises the *right* exception with `pytest.raises` — as
important as testing the happy path.

In [ ]:
%%ipytest

import pytest

def parse_amount(s):
    value = float(s)
    if value < 0:
        raise ValueError('amount must be non-negative')
    return value

def test_rejects_negative():
    with pytest.raises(ValueError):
        parse_amount('-5')

def test_rejects_garbage():
    with pytest.raises(ValueError):
        parse_amount('abc')

> **Running tests outside notebooks:** put tests in `tests/test_*.py` and run
> `uv run pytest` at the project root. The notebook uses `ipytest` only so the
> examples are self-contained.

### Recap

Tests are `test_*` functions with `assert`s; `parametrize` covers many cases;
fixtures supply reusable setup; `pytest.raises` checks error paths. Test your
pure transforms first — they carry the most risk. Next: performance and memory.